# Evaluacija ConvNeXt modela

Ova sveska služi za evaluaciju prethodno istreniranog ConvNeXt modela na test skupu slika. Najpre se učitavaju potrebne biblioteke i određuje uređaj za izvršavanje modela (GPU ili CPU). Zatim se definišu transformacije test slika, učitava test skup i formira DataLoader. Nakon toga se kreira ConvNeXt Tiny model sa 200 izlaznih klasa i učitavaju se prethodno sačuvani trenirani parametri modela. Model se postavlja u evaluacioni režim i koristi se za predikciju klasa test slika. Na kraju se računaju tačnost (Top-1), gubitak (loss), macro precision, macro recall, macro F1 i brzina obrade slika, a dobijeni rezultati se čuvaju u JSON datoteci radi dalje analize.

In [ ]:
import time
import copy
import pandas as pd
from pathlib import Path
from sklearn.metrics import precision_recall_fscore_support
import torch
import torch.nn as nn
import timm
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import (
    convnext_tiny,
    ConvNeXt_Tiny_Weights)
import json

In [2]:
def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

def bind_gpu(data):
    device = get_device()
    if isinstance(data, (list, tuple)):
        return [bind_gpu(data_elem) for data_elem in data]
    else:
        return data.to(device, non_blocking=True)
device=get_device()

In [3]:
IMG_SIZE=224
NUM_WORKERS=8
DATA_PATH ="tiny-imagenet-200-modified/test_holdout/"
BATCH_SIZE=128

In [5]:
test_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])

test_dataset = datasets.ImageFolder(DATA_PATH, transform = test_transform)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=False)



In [6]:
model = convnext_tiny()
model.classifier[2] = nn.Linear(768, 200)

checkpoint = torch.load("trained_models/convnext/convnext_best_model.pth", map_location=device)

state_dict = {key.replace("model.", "", 1): value for key, value in checkpoint.items()}

model.load_state_dict(state_dict)
model = model.to(device)
model.eval()

lossFunction = nn.CrossEntropyLoss()

total_loss = 0.0
total = 0
correct_top1 = 0
top1_predicts = []
true_labels = []

with torch.no_grad():
    start_time = time.time()

    for inputs, labels in test_loader:

        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)

        loss = lossFunction(outputs,labels)

        total_loss += loss.item()* inputs.size(0)

        
        top1 = outputs.argmax(dim=1)
        predicted = torch.argmax(outputs,dim=1)

        correct_top1 += (predicted == labels).sum().item()

        top1_hit = (top1 == labels).sum().item()

        top1_predicts.extend(top1.cpu().tolist())
        true_labels.extend(labels.cpu().tolist())
        total += inputs.size(0)
        
    total_time = time.time() - start_time
    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels, top1_predicts, average="macro", zero_division=0
    )

test_metrics = {
        "top1_acc": correct_top1 / total,
        "loss": total_loss / total,
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1,
        "throughput_img_s": total / total_time
    }




In [7]:
with open("eval_convnext.json", "w", encoding="utf-8") as fajl:
    json.dump(test_metrics, fajl, indent=4)